In [1]:
from spin_lattices import KagomeLattice, SpinLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from boolean_analysis import (
    BooleanFourierAnalyzer,
    keep_largest_n,
    keep_everything,
    ScorerType,
    get_scorer,
    SignalOption,
    AmplitudeMedianBinSignalKind,
    SignSignalKind,
    AmplitudeSignalKind,
    SignalKind,
)
from boolean_fourier_learner import BooleanFourierLearner

from pathlib import Path
import numpy as np
import pandas as pd
import lattice_symmetries as ls
import matplotlib.pyplot as plt
from heisenberg_hamiltonians import batched_state_info_df
from itertools import product
import numpy.typing as npt
from tqdm import tqdm
import seaborn as sns
import parse

from parity import popcount, parity

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data

import pickle
from spin_nn import SpinNN
import datetime

ground_state_cache_dir = Path("groundstates")
fourier_learners_cache_dir = Path("fourier_learners_cache")
experiments_dir = Path("experiments") / "kagome-24-nn-2023-01-27"
experiments_dir.mkdir(parents=True, exist_ok=True)

2023-01-27 18:59:04.178 | DEBUG    | lattice_symmetries:__init__:49 - Initializing Haskell runtime...
2023-01-27 18:59:04.185 | DEBUG    | lattice_symmetries:__init__:51 - Initializing Chapel runtime...
2023-01-27 18:59:04.234 | DEBUG    | lattice_symmetries:__init__:53 - Setting Python exception handler...
[Debug]   [LOCALE0]   Initializing chpl_kernels ...
set_python_exception_handler ...


In [2]:
class FC1SpinNN(SpinNN):
    def __init__(self, lattice: SpinLattice, hidden_size: int):
        super().__init__(lattice)
        input_size = lattice.number_spins
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 2)

    def forward(self, inp: npt.NDArray[np.uint64]) -> torch.Tensor:
        x = self.preprocess(inp)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return  self.postprocess(x)
        

In [3]:
J2 = 0
system = HeisenbergJ1J2(
    lattice=KagomeLattice(width=2, height=4),
    J1=1,
    J2=J2,
    use_symmetries=True,
    spin_inversion=1,
    ground_state_cache_dir=ground_state_cache_dir,
    show_progress=True,
)
system.get_eigenstates(1)

number_spins=24
Symmetry group contains 16 elements
Hilbert space dimension is 85662
Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-0.0-True-1-1.pickle
Ground state energy is -46.1341392314


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


(array([-46.13413923]),
 array([[ 3.03997239e-06],
        [-4.36694621e-06],
        [ 3.03997239e-06],
        ...,
        [-1.63915932e-03],
        [-1.19165754e-03],
        [-1.08085359e-03]]))

In [4]:
df = (
    system.get_df_ground_state(
        canonical_basis=True,
    )
    .assign(
        sign=(lambda df: np.sign(df["eigenstate_coeff"])),
        prob=(lambda df: np.abs(df["eigenstate_coeff"]) ** 2),
    )
    .assign(y=lambda df: (df["sign"] == 1).astype(int))
)

In [5]:
eps_train = 2e-3
val_eps = 1e-2
test_eps = 1e-2
batch_size = 64

df_train = df.sample(frac=eps_train, weights="prob")
df_val = df.sample(frac=val_eps, weights="prob")
df_test = df.sample(frac=test_eps, weights="prob")

n_batches = int(np.ceil(len(df_train) / batch_size))
epochs = 20000


In [6]:
net = FC1SpinNN(lattice=system.lattice, hidden_size=64)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)


In [7]:
def get_inputs_and_labels(df: pd.DataFrame) -> tuple[npt.NDArray[np.uint64], torch.Tensor, torch.Tensor]:
    X = np.asarray(df.index, dtype="uint64")
    y = torch.tensor(df["y"].values.astype("int8"), dtype=torch.long)
    probs = torch.tensor(df["prob"].values.astype("float"), dtype=torch.float32)
    return X, y, probs

inputs_val, labels_val, probs_val = get_inputs_and_labels(df_val)


In [8]:
def evaluate(net, inputs, labels, probs):
    with torch.no_grad():
        outputs = net(inputs)
        _, predicted = torch.max(outputs.data, 1)
        correct = (predicted == labels).sum().item()
        accuracy = correct / len(labels)

        sign_overlap = (
            ((predicted * 2 - 1) * (labels * 2 - 1) * probs).sum() / probs.sum()
        ).item()
        return accuracy, sign_overlap


In [12]:
for epoch in range(epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    i = None
    loss = None

    for i in range(n_batches):
        data = df_train.iloc[i * batch_size : (i + 1) * batch_size]
        inputs, labels, probs = get_inputs_and_labels(data)

        # zero the parameter gradients
        optimizer.zero_grad()

        print(">>> Finding outputs")
        # forward + backward + optimize
        outputs = net(inputs)
        print(">>> Calculating loss")
        loss = criterion(outputs, labels)
        loss.backward()

        print(">>> Optimizing")
        optimizer.step()

    print(f"[{epoch + 1}, {i}] loss: {loss}")

    accuracy, sign_overlap = evaluate(net, inputs, labels, probs)
    print(f"Test set: accuracy: {100 * accuracy} %, sign overlap: {sign_overlap}")

    accuracy_val, sign_overlap_val = evaluate(net, inputs_val, labels_val, probs_val)
    print(f"Validation set: accuracy: {100 * accuracy_val} %, sign overlap: {sign_overlap_val}")


>>> Finding outputs
Preprocessing...
Making orbits...
Finding extended_states...
Unpacking configurations...
>>> Calculating loss
>>> Optimizing
>>> Finding outputs
Preprocessing...
Making orbits...
Finding extended_states...
Unpacking configurations...
>>> Calculating loss
>>> Optimizing
>>> Finding outputs
Preprocessing...
Making orbits...
Finding extended_states...
Unpacking configurations...
>>> Calculating loss
>>> Optimizing
>>> Finding outputs
Preprocessing...
Making orbits...
Finding extended_states...
Unpacking configurations...
>>> Calculating loss
>>> Optimizing
>>> Finding outputs
Preprocessing...
Making orbits...
Finding extended_states...
Unpacking configurations...
>>> Calculating loss
>>> Optimizing
>>> Finding outputs
Preprocessing...
Making orbits...
Finding extended_states...
Unpacking configurations...
>>> Calculating loss
>>> Optimizing
>>> Finding outputs
Preprocessing...
Making orbits...
Finding extended_states...
Unpacking configurations...
>>> Calculating loss


KeyboardInterrupt: 

In [11]:
n_batches

85

In [ ]:
from utils import make_unpacked_configurations

In [14]:
rel = (
    net.state_info_df.reset_index()
    .rename(columns={"index": "state"})
    .merge(
        net.state_info_df.reset_index().rename(columns={"index": "input_state"})[
            ["input_state", "representative"]
        ],
        on="representative",
        how="inner",
    )
)
